# Descarga de Datos

En esta libreta se gestionara la descarga de los datos para nuestro analisis.

## Introducción.
Como primer paso haremos la selección del conjunto de datos a analizar. En nuestro proyecto nos centraremos en la parte Agrícola. Sin embargo, es posible acceder a la información Pecuaria.

Para la descarga de datos usaremos las librerias BeautifulSoup and Requests.

In [8]:
import requests
from bs4 import BeautifulSoup
import yarl
import re
import pandas as pd


# Pagina principal de los Datos.
MAIN_URL = yarl.URL("https://nube.agricultura.gob.mx")
DGSIAP_URL = MAIN_URL / "datosAbiertos/"
DATA_URL = MAIN_URL / "index.php"


print("Bienvenido al script de descarga de datos abiertos del DGSIAP.")
info = {
    "Agricola": "Agricola.php",
    "Pecuaria": "Ganadera.php",
}

TEST = 1

if not TEST:
    value = input(f"Ingrese la categoria de los datos a consultar. {" o ".join(info.keys())}: ")

    try:
        DGSIAP = DGSIAP_URL / info[value]
    except KeyError:
        print("Categoria no valida. Por favor, ingrese una categoria valida.")
        exit(1)
else:
    DGSIAP = DGSIAP_URL / info["Agricola"]


response = requests.get(DGSIAP)
if response.status_code != 200:
    print(f"Error al acceder a la pagina: {response.status_code}")
    exit(1)

dgsiap = BeautifulSoup(response.content, "html.parser")

Bienvenido al script de descarga de datos abiertos del DGSIAP.


Notemos que una vez definida la categoria.
* Agricola
* Pecuaria.

BeatifulSoup nos permite explorar la pagina con HTML. Elemento que aprovecharemos para encontrar la información que requeriremos.

In [2]:
content = dgsiap.find("main")
sections = content.find_all("section", recursive=False)
tree = sections[0]


Vemos que la primera sección corresponde al arbol definido en la propia pagina. Notemos que de ahi podemos extraer información relevante.

In [3]:
for t in tree.find_all("li"):
    if t.text!="":
        print(t.text)

Dirección General del Servicio de Información Agroalimentaria y Pesquera
Datos abiertos
Información agrícola


In [4]:
intro = sections[1]
raw_intro=list()
for t in intro.find_all("div", recursive=False):
    if t.text!="":
        raw_block=(t.text).split("\t")
        for raw in raw_block:
            if raw.strip() != "":
                raw_intro.append(raw)

print(raw_intro)

['Datos abiertos', 'Estadística de Producción Agrícola\r\n', 'Los usuarios de la información podrán disponer de los datos anualizados de estadística básica agrícola a nivel nacional, estatal, por Distrito de Desarrollo Rural y municipal, la cual será de gran utilidad para el análisis del subsector y la toma de decisiones públicas y privadas que contribuyan al desarrollo rural sustentable del país.\r\n']


In [5]:
last = sections[2]
search_title_section=last.find("div")
last_data = last.find("div", {"class":"bento-card h-100"})
last_dict = last.find("div", {"class":"dict-section"})
ld_bs = last_dict.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
print(last_dict)

<div class="dict-section" style="--dark-teal:var(--cherry-on)">
<div class="row align-items-center position-relative" style="z-index:5;">
<div class="col-12 col-md-6 mb-5 mb-md-0">
<h2>Diccionario de datos</h2>
<p>Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.</p>
<div class="d-flex flex-wrap justify-content-end" style="gap:1rem;">
<a class="btn-dict-primary text-decoration-none" href="/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7" target="_blank">
<i class="fa-duotone fa-thin fa-file-spreadsheet fa-xl"></i>
												Descargar XLSX
											</a>
</div>
</div>
<div class="col-12 col-md-6">
<div class="meta-card">
<h4>Metadatos Principales</h4>
<div class="meta-row">
<span class="label">Actualizado</span>
<span class="value">12 agosto 2026</span>
</div>
<div class="meta-row">
<span class="label">Cobertura 2003 - 2025</span>
<span class="value">Nivel Municipal</span>
</div>
</div>
</div>
</div>
</div>


In [22]:
catalog, metadata = ld_bs[0], ld_bs[1]

print(catalog)
b_title = catalog.find("h2").text
b_description=catalog.find("p").text
b_download = catalog.find("a")['href']
print(b_title, b_description, b_download,"\n")

print(DATA_URL)

DOWNLOAD_DATA=DATA_URL.update_query(view=b_download.split("view=")[-1])

file = requests.get(DOWNLOAD_DATA)
file_h = requests.head(DOWNLOAD_DATA)
print(file)
print(file_h)

for u, v in file_h.headers.items():
    try:
        if v.index("filename"):
            print(u,"A", v)
    except:
        pass


<div class="col-12 col-md-6 mb-5 mb-md-0">
<h2>Diccionario de datos</h2>
<p>Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.</p>
<div class="d-flex flex-wrap justify-content-end" style="gap:1rem;">
<a class="btn-dict-primary text-decoration-none" href="/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7" target="_blank">
<i class="fa-duotone fa-thin fa-file-spreadsheet fa-xl"></i>
												Descargar XLSX
											</a>
</div>
</div>
Diccionario de datos Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos. /index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7 

https://nube.agricultura.gob.mx/index.php
<Response [200]>
<Response [200]>
Content-Disposition A attachment; filename="Diccionario_agricola_2003_a_2025.xlsx"


## Ultima Publicación.

Para un analisis rápido podemos obtener la información más reciente. 


In [7]:
search_last = dgsiap.find("section", {"class":"content-box pt-0 mb-5"})
print(search_last)

<section class="content-box pt-0 mb-5">
<div class="row">
<div class="col-sm">
<div class="h5 font-weight-bold section-title">
								Última publicación
							</div>
<hr class="red"/>
</div>
</div>
<div class="row" style="margin-top:-45px;">
<div class="col-sm-6 mb-4">
<div class="bento-card h-100">
<div class="d-flex justify-content-between align-items-start mb-4">
<span class="badge-dataset text-center">Nivel Municipal</span>
<span class="badge-new text-center">RECIENTE 2025</span>
</div>
<h3 style="font-size:1.35rem;margin-bottom:.6rem;">Cierre de la Producción Agrícola 2025</h3>
<p class="mb-5 header-content-title" style="color:var(--on-surface-var);line-height:1.6;">
									Cifras definitivas a nivel municipal correspondiente al año 2025.
								</p>
<div class="d-flex align-items-center" style="gap:2.5rem;flex-wrap:wrap;">
<a class="btn btn-cherry text-decoration-none rounded-pill px-4 py-2" href="/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&amp;ANIO=2025" target="_bl